# 01 — Feature Extraction, Sampling, and DFT Input Preparation

This notebook is part of a minimal working example demonstrating a
**coupled sampling–training framework** for MLIP development.

Using a small dataset, the example shows how the choice of sampling strategy
affects what structures are selected from an AIMD candidate pool — and
ultimately how it impacts model quality after fine-tuning.
By running both notebooks you can directly compare Random, DIRECT, and LCMD
sampling within the same training pipeline.

This notebook covers:

1. **Feature extraction** — M3GNet structural features → HDF5  
2. **PCA** — dimension reduction with Kaiser's rule  
3. **Sampling** — 100 structures via Random / DIRECT / LCMD  
4. **DFT input generation** — POSCAR / INCAR / KPOINTS / POTCAR (MatPESStaticSet)  

After running this notebook, submit the VASP jobs, collect DFT outputs into
ExtXYZ format, then continue with `02_training_and_evaluation.ipynb`.

---
### Acknowledgements

This pipeline uses
[maml](https://github.com/materialsvirtuallab/maml) for M3GNet feature extraction and
[SevenNet](https://github.com/MDIL-SNU/SevenNet) for MLIP training.
Code was written with the assistance of [Claude](https://claude.ai) (Anthropic).

---
### Directory layout
```
data/1_example_data/
    v1_dataset/
        initial_200fs_500.xyz              ← center (foundational) — first 400 used
        augmented_10fs_10000.xyz            ← candidate pool
        pca_model/
            m3gnet_pca_pca_model.json      ← pre-trained PCA model
            initial_200fs_500_reduced.h5
            augmented_10fs_10000_reduced.h5
    v1_models/                             ← all outputs written here
        sampling/
        vasp_inputs/
        results/
```

In [ ]:
import os, sys
import numpy as np
sys.path.insert(0, os.path.abspath('.'))

from src.features    import extract_features, train_pca
from src.sampling    import load_reduced_h5, random_sampling, direct_sampling, lcmd_sampling, save_selection
from src.vasp_inputs import generate_vasp_inputs
from src.plotting    import plot_pca_space, plot_pca_variance

# dataset and output root directories
DATASET_DIR  = 'data/1_example_data/v1_dataset'
MODELS_DIR   = 'data/1_example_data/v1_models'

# input files
CENTER_XYZ = f'{DATASET_DIR}/initial_200fs_500.xyz'
#CAND_XYZ   = f'{DATASET_DIR}/augmented_10fs_10000.xyz'

# pre-computed PCA-reduced feature files
CENTER_H5  = f'{DATASET_DIR}/pca_model/initial_200fs_500_reduced.h5'
CAND_H5    = f'{DATASET_DIR}/pca_model/augmented_10fs_10000_reduced.h5'
PCA_MODEL  = f'{DATASET_DIR}/pca_model/m3gnet_pca_pca_model.json'

# output directories
SAMPLING_DIR = f'{MODELS_DIR}/sampling'
VASP_DIR     = f'{MODELS_DIR}/vasp_inputs'
RESULTS_DIR  = f'{MODELS_DIR}/results'

# VASP POTCAR library — update to your local path
POTCAR_ROOT = '/opt/vasp/Potential/potpaw_PBE.54'

os.makedirs(SAMPLING_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,  exist_ok=True)

## Step 1 — M3GNet feature extraction

Each structure in the ExtXYZ file is encoded into a fixed-length descriptor
vector using a pre-trained M3GNet model. The vectors are saved to HDF5 for
efficient downstream processing.

Pre-computed feature files are already provided in `pca_model/`.
Run this cell only if you want to regenerate them from scratch.

In [ ]:
# encode all structures in each xyz file into M3GNet descriptor vectors
extract_features(
    xyz_file  = CENTER_XYZ,
    output_h5 = f'{DATASET_DIR}/initial_200fs_500_features.h5',
    batch_size = 32,
    n_jobs     = 4,
)
extract_features(
    xyz_file  = CAND_XYZ,
    output_h5 = f'{DATASET_DIR}/augmented_10fs_10000_features.h5',
    batch_size = 32,
    n_jobs     = 4,
)

## Step 2 — PCA dimension reduction

High-dimensional M3GNet descriptors are compressed with PCA.
The number of components is chosen by Kaiser's rule (eigenvalue > 1),
which retains components that explain at least as much variance as a
single original feature.

Pre-computed reduced files are already provided in `pca_model/`.
Run this cell only if you want to retrain the PCA model.

In [ ]:
# fit PCA jointly on center + candidate features, then project both sets
train_pca(
    h5_files           = [
        f'{DATASET_DIR}/initial_200fs_500_features.h5',
        f'{DATASET_DIR}/augmented_10fs_10000_features.h5',
    ],
    output_dir         = f'{DATASET_DIR}/pca_model',
    model_name         = 'm3gnet_pca',
    save_reduced       = True,
    reduced_output_dir = f'{DATASET_DIR}/pca_model',
)

In [ ]:
# plot cumulative and per-component explained variance with Kaiser cutoff
plot_pca_variance(
    pca_model_json   = PCA_MODEL,
    output_path      = f'{RESULTS_DIR}/pca_variance.png',
    extra_components = 5,
)

## Step 3 — Load reduced features

In [ ]:
# load PCA-reduced features for the center and candidate sets
center_feats_all, center_ids_all = load_reduced_h5(CENTER_H5)
cand_feats, cand_ids             = load_reduced_h5(CAND_H5)

# use the first 400 structures as the foundational (center) set
center_feats = center_feats_all[:400]
center_ids   = center_ids_all[:400]

print(f'Center   : {len(center_feats):,} structures × {center_feats.shape[1]} PCs')
print(f'Candidate: {len(cand_feats):,} structures × {cand_feats.shape[1]} PCs')

## Step 4 — Sampling (100 structures per method)

Three methods are compared to illustrate how data selection strategy
affects the distribution of selected structures in feature space —
and ultimately the quality of the fine-tuned model.

| Method | Description |
|--------|-------------|
| **Random** | Uniform random selection (baseline) |
| **DIRECT** | Birch clustering — one centroid representative per cluster |
| **LCMD**   | Greedy farthest-point seeded from the foundational dataset |

In [ ]:
# Random: draw 100 structures uniformly at random from the candidate pool
idx_random = random_sampling(cand_feats, n_samples=100, seed=42)

save_selection(
    idx_random, cand_ids,
    json_out = f'{SAMPLING_DIR}/random_selected.json',
    method   = 'random',
    xyz_src  = CAND_XYZ,
    xyz_out  = f'{SAMPLING_DIR}/random_selected.xyz',
)

In [ ]:
# DIRECT: Birch-cluster the candidate pool, pick the centroid-nearest member
# of each cluster; bisection search adjusts the threshold until ~100 clusters form
idx_direct, best_thresh = direct_sampling(
    cand_feats,
    n_samples      = 100,
    threshold_init = 1.0,
    tol            = 0.05,
)
save_selection(
    idx_direct, cand_ids,
    json_out   = f'{SAMPLING_DIR}/direct_selected.json',
    method     = 'direct',
    xyz_src    = CAND_XYZ,
    xyz_out    = f'{SAMPLING_DIR}/direct_selected.xyz',
    extra_info = {'birch_threshold': best_thresh},
)

In [ ]:
# LCMD: greedy farthest-point — at each step pick the candidate with the
# largest minimum distance to the current selected set, seeded from center_feats;
# distance updates use BLAS gemv: dist²(x,y) = ||x||² + ||y||² - 2x·y
idx_lcmd = lcmd_sampling(
    cand_feats,
    center_feats,
    n_samples = 100,
)
save_selection(
    idx_lcmd, cand_ids,
    json_out = f'{SAMPLING_DIR}/lcmd_selected.json',
    method   = 'lcmd',
    xyz_src  = CAND_XYZ,
    xyz_out  = f'{SAMPLING_DIR}/lcmd_selected.xyz',
)

In [ ]:
print(f'Random : {len(idx_random)} structures')
print(f'DIRECT : {len(idx_direct)} structures  (Birch threshold = {best_thresh:.4f})')
print(f'LCMD   : {len(idx_lcmd)} structures')

In [ ]:
# visualise each method's selection overlaid on the full candidate PCA space
for method, idx in [('random', idx_random), ('direct', idx_direct), ('lcmd', idx_lcmd)]:
    plot_pca_space(
        all_feats    = cand_feats,
        selected     = {method: cand_feats[idx]},
        center_feats = center_feats,
        output_path  = f'{RESULTS_DIR}/pca_{method}.png',
        title        = f'PCA — {method.capitalize()} (n=100)',
    )

# overlay all three methods in a single figure for direct comparison
plot_pca_space(
    all_feats    = cand_feats,
    selected     = {
        'random': cand_feats[idx_random],
        'direct': cand_feats[idx_direct],
        'lcmd'  : cand_feats[idx_lcmd],
    },
    center_feats = center_feats,
    output_path  = f'{RESULTS_DIR}/pca_all_methods.png',
    title        = 'PCA — all sampling methods (n=100 each)',
)

## Step 5 — Generate VASP input files

For each selected structure, write a complete VASP static calculation input set
(POSCAR / INCAR / KPOINTS / POTCAR) using pymatgen's `MatPESStaticSet`.

Update `POTCAR_ROOT` above to point to your local POTCAR library.

In [ ]:
# write VASP inputs for each sampling method into separate subdirectories
for method, idx in [('random', idx_random), ('direct', idx_direct), ('lcmd', idx_lcmd)]:
    print(f'\n── {method.upper()} ({len(idx)} structures) ──')
    generate_vasp_inputs(
        xyz_file       = CAND_XYZ,
        output_root    = f'{VASP_DIR}/{method}',
        potcar_root    = POTCAR_ROOT,
        kpoint_density = 100,
        n_groups       = 1,
        indices        = idx,
    )

In [ ]:
print('Done.')
print(f'VASP inputs  →  {VASP_DIR}/{{random,direct,lcmd}}/')
print(f'Sampling XYZ →  {SAMPLING_DIR}/')
print(f'Figures      →  {RESULTS_DIR}/')
print()
print('Next: run VASP, collect DFT outputs into ExtXYZ,')
print('then open 02_training_and_evaluation.ipynb')